# 02 · Feature Engineering — HDB Rent

This notebook generates three groups of row-level features that do not contain rental labels (core / building / amenities), along with a separate monthly macroeconomic feature table.
**It does not fit imputers, encoders, or scalers, and it does not generate target mean encodings from the full dataset.** Labels are saved separately to prevent accidental inclusion in model inputs.

Input: The eight raw CSV files in `data/raw/`.
Output: `data/processed/`.


In [1]:
from pathlib import Path
import os
import json
import re
import hashlib
import platform
import importlib.metadata as package_metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

SEED = 42
rng = np.random.default_rng(SEED)
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', context='notebook')

def locate_project_root():
    override = os.environ.get('HDB_PROJECT_ROOT')
    candidates = [Path(override).expanduser().resolve()] if override else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'data' / 'raw' / 'train.csv').is_file():
            return candidate
    raise FileNotFoundError('Put train.csv and test.csv in PROJECT/data/raw/. Open this notebook from PROJECT or PROJECT/notebooks/.')

ROOT = locate_project_root()
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
OUTPUTS = ROOT / 'outputs'
for directory in [PROCESSED, OUTPUTS]:
    directory.mkdir(parents=True, exist_ok=True)

FILES = ['train.csv', 'test.csv', 'sg-hdb-block.csv', 'sg-mrt-stations.csv',
         'sg-shopping-malls.csv', 'sg-schools.csv', 'sg-coe-prices.csv', 'sg-stock-prices.csv']

def raw_path(name):
    candidates = [RAW / name, RAW / 'auxiliary-data' / name]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f'Missing {name}: expected in {RAW} or {RAW / "auxiliary-data"}.')

def load_raw(name):
    # Preserve identifiers such as 090034 and 321A as text.
    return pd.read_csv(raw_path(name), dtype={'BLOCK': 'string', 'POSTAL_CODE': 'string'})

def clean_text(series):
    return series.astype('string').str.normalize('NFKC').str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

def canonical_flat_type(series):
    return clean_text(series).str.replace(r'[\s-]*room\b', '-room', regex=True)

def canonical_keys(frame):
    out = frame.copy()
    for column in ['TOWN', 'BLOCK', 'STREET']:
        if column in out:
            out[column] = clean_text(out[column])
    if 'FLAT_TYPE' in out:
        out['FLAT_TYPE'] = canonical_flat_type(out['FLAT_TYPE'])
    return out

def parse_amount(series):
    return pd.to_numeric(series.astype('string').str.replace(r'[$,\s]', '', regex=True), errors='coerce')

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
def parse_coe_date(year, month):
    month_map = {
        'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4,
        'may': 5, 'jun': 6, 'jul': 7, 'aug': 8,
        'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
    }

    month_number = (
        month.astype('string')
        .str.strip()
        .str.lower()
        .map(month_map)
    )

    if month_number.isna().any():
        invalid_values = month.loc[month_number.isna()].unique()
        raise ValueError(f'Unrecognized month values: {invalid_values}')

    return pd.to_datetime({
        'year': year,
        'month': month_number,
        'day': 1
    })

print('Project:', ROOT)
print('Python:', platform.python_version())


Project: d:\AppData\NUS\Sem1\5228 Knowledge Discovery and Data Mining\cs5228-hdb-rent-prediction
Python: 3.12.15


## 1. Loading, Normalization, and Row-order Preservation

`row_id` is the zero-based row index in the original CSV. It is used only to verify and restore row order, and is not included as a feature. Rental labels are saved separately.


In [2]:
train_raw = load_raw('train.csv')
test_raw = load_raw('test.csv')
assert 'MONTHLY_RENT' not in test_raw
assert train_raw.MONTHLY_RENT.gt(0).all()
targets = pd.DataFrame({'row_id': np.arange(len(train_raw)), 'monthly_rent': train_raw.MONTHLY_RENT.to_numpy()})

def normalize_rent_table(raw):
    out = canonical_keys(raw.drop(columns='MONTHLY_RENT', errors='ignore'))
    for column in ['FLAT_MODEL', 'FURNISHED']:
        out[column] = clean_text(out[column])
    out.columns = out.columns.str.lower()
    out.insert(0, 'row_id', np.arange(len(out)))
    out['rent_approval_date'] = pd.to_datetime(out.rent_approval_date, format='%Y-%m', errors='raise')
    return out

train_base = normalize_rent_table(train_raw)
test_base = normalize_rent_table(test_raw)
constant_raw_columns = [c for c in train_base if c not in ['row_id', 'rent_approval_date']
                        and train_base[c].nunique(dropna=False) <= 1]
print('Constants excluded from the feature lists:', constant_raw_columns)
print('Train/test rows:', len(train_base), len(test_base))


Constants excluded from the feature lists: ['furnished', 'fee']
Train/test rows: 150000 50000


## 2. Property Attributes and Temporal Features

The month index helps models capture trends, while sin/cos features represent the annual cycle. Tree models generally do not naturally extrapolate temporal trends, so Ridge is also included in the subsequent comparison.

`lease_age` and `building_age` have different meanings. Executive flats are not forced into the room-count mapping used for standard n-room flats.


In [3]:
def add_core_features(frame):
    out = frame.copy()
    date = out.rent_approval_date
    out['approval_year'] = date.dt.year
    out['approval_month'] = date.dt.month
    out['month_index'] = (date.dt.year - 2021) * 12 + date.dt.month - 1
    out['month_sin'] = np.sin(2 * np.pi * date.dt.month / 12)
    out['month_cos'] = np.cos(2 * np.pi * date.dt.month / 12)
    out['lease_age'] = date.dt.year - out.lease_commence_date
    out['lease_age_invalid'] = out.lease_age.lt(0).astype('int8')
    out.loc[out.lease_age.lt(0), 'lease_age'] = np.nan
    out['log_floor_area'] = np.log1p(out.floor_area_sqm.where(out.floor_area_sqm.gt(0)))
    out['flat_room_number'] = pd.to_numeric(out.flat_type.str.extract(r'^(\d+)-room$', expand=False), errors='coerce')
    return out

train_base = add_core_features(train_base)
test_base = add_core_features(test_base)
core_features = ['town', 'flat_type', 'flat_model', 'floor_area_sqm', 'lease_commence_date',
                 'approval_year', 'approval_month', 'month_index', 'month_sin', 'month_cos',
                 'lease_age', 'lease_age_invalid', 'log_floor_area', 'flat_room_number']
# A constant in training carries no learnable effect; exclude it without consulting test labels.
core_features = [c for c in core_features if c not in constant_raw_columns]
display(train_base[core_features].head())


,town,flat_type,flat_model,floor_area_sqm,lease_commence_date,approval_year,approval_month,month_index,month_sin,month_cos,lease_age,lease_age_invalid,log_floor_area,flat_room_number
0,punggol,4-room,model a,93.0,2012,2021,1,0,0.5,0.866025,9.0,0,4.543295,4
1,sengkang,5-room,improved,110.0,2001,2021,1,0,0.5,0.866025,20.0,0,4.709530,5
2,queenstown,3-room,improved,65.0,1976,2021,1,0,0.5,0.866025,45.0,0,4.189655,3
3,sengkang,4-room,model a,96.0,2010,2021,1,0,0.5,0.866025,11.0,0,4.574711,4
4,sembawang,4-room,premium apartment,102.0,2001,2021,1,0,0.5,0.866025,20.0,0,4.634729,4


## 3. Joining Building Information

Leading zeros in postal codes are preserved. All joins are checked for cardinality and row-order preservation. Missing building information remains NaN and is handled by preprocessors fitted within each training fold.


In [4]:
blocks = canonical_keys(load_raw('sg-hdb-block.csv'))
blocks.columns = blocks.columns.str.lower()
for column in ['subzone', 'planning_area', 'region']:
    blocks[column] = clean_text(blocks[column])
# deal with NIL postal codes
postal = clean_text(blocks['postal_code'])
postal = postal.where(postal.str.fullmatch(r'[0-9]{1,6}', na=False))
blocks['postal_code'] = postal.str.zfill(6)
blocks['postal_sector'] = blocks.postal_code.str[:2]
# group by 'town + street'
grouped = blocks.groupby(['town', 'street'])['postal_sector']
sector_count = grouped.transform('nunique')
sector_candidate = grouped.transform('first')
# Fill in missing postal_sector values for blocks where there is only one unique postal_sector for that town + street combination.
fill_mask = blocks.postal_sector.isna() & sector_count.eq(1)
blocks['postal_sector_imputed'] = fill_mask.astype('int8')
blocks.loc[fill_mask, 'postal_sector'] = sector_candidate.loc[fill_mask]
keys = ['town', 'block', 'street']
assert not blocks.duplicated(keys).any(), 'Resolve duplicate block keys before merging.'
blocks['building_id'] = blocks[keys].agg('|'.join, axis=1)

def add_building_features(frame):
    out = frame.merge(blocks, on=keys, how='left', sort=False, validate='many_to_one', indicator=True)
    assert len(out) == len(frame)
    assert np.array_equal(out.row_id.to_numpy(), frame.row_id.to_numpy())
    out['block_metadata_missing'] = out['_merge'].ne('both').astype('int8')
    out = out.drop(columns='_merge')
    out['building_age'] = out.rent_approval_date.dt.year - out.year_completed
    out['building_age_invalid'] = out.building_age.lt(0).astype('int8')
    out.loc[out.building_age.lt(0), 'building_age'] = np.nan
    return out

train_enriched = add_building_features(train_base)
test_enriched = add_building_features(test_base)
building_features = ['street', 'building_id', 'postal_sector', 'postal_sector_imputed', 'latitude', 'longitude', 'max_floor',
                     'year_completed', 'building_age', 'building_age_invalid', 'subzone',
                     'planning_area', 'region', 'block_metadata_missing']
display(pd.DataFrame({'split': ['train', 'test'],
    'block_join_coverage': [1 - train_enriched.block_metadata_missing.mean(),
                          1 - test_enriched.block_metadata_missing.mean()]}))


,split,block_join_coverage
0,train,1.0
1,test,1.0


## 4. Landmark Distances: Compute for Unique Buildings, Then Join Back to Rental Records

Distances are calculated in km using BallTree + haversine. Counts within a radius use actual distance thresholds in km.
Stations with the same name are consolidated using median coordinates to avoid counting interchange stations multiple times.

`mrt_open_snapshot` explicitly represents the status snapshot in the supplied file; it does not imply that all these stations were already open during each historical rental month.
`mrt_planned_snapshot` is calculated separately and is not combined with open stations. Primary-school features include primary schools and mixed-level schools that offer p1.


In [9]:
from sklearn.neighbors import BallTree
EARTH_RADIUS_KM = 6371.0088

def normalize_poi(raw):
    out = raw.copy()
    out.columns = out.columns.str.lower()
    out['name'] = clean_text(out['name'])
    return out

mrt = normalize_poi(load_raw('sg-mrt-stations.csv'))
mrt['status'] = clean_text(mrt.status)
malls = normalize_poi(load_raw('sg-shopping-malls.csv'))
schools = normalize_poi(load_raw('sg-schools.csv'))
school_level = clean_text(schools.mainlevel_code)
primary = schools[school_level.str.contains(r'primary|p1', regex=True, na=False)]
poi_sets = {
    'mrt_open_snapshot': mrt[mrt.status.eq('open')],
    'mrt_planned_snapshot': mrt[mrt.status.eq('planned')],
    'mall_snapshot': malls,
    'primary_school_snapshot': primary,
}
locations = blocks[['building_id', 'latitude', 'longitude']].copy()
amenity_features = []
poi_summary = []
valid_locations = locations[['latitude', 'longitude']].notna().all(axis=1)
location_radians = np.radians(locations.loc[valid_locations, ['latitude', 'longitude']].to_numpy())
for name, poi in poi_sets.items():
    points = poi.dropna(subset=['latitude', 'longitude']).groupby('name', as_index=False)[['latitude', 'longitude']].median()
    poi_summary.append({'source': name, 'input_rows': len(poi), 'unique_locations': len(points)})
    nearest_column = f'dist_{name}_km'
    locations[nearest_column] = np.nan
    amenity_features.append(nearest_column)
    if len(points):
        tree = BallTree(np.radians(points[['latitude', 'longitude']].to_numpy()), metric='haversine')
        distances, _ = tree.query(location_radians, k=1)
        locations.loc[valid_locations, nearest_column] = distances[:, 0] * EARTH_RADIUS_KM
    for radius_km in [.5, 1.0]:
        column = f'count_{name}_{int(radius_km * 1000)}m'
        locations[column] = np.nan
        if len(points):
            locations.loc[valid_locations, column] = tree.query_radius(
                location_radians, r=radius_km / EARTH_RADIUS_KM, count_only=True)
        amenity_features.append(column)

# A named location proxy from the provided MRT file; no external coordinates are hard-coded.
reference = mrt[mrt.name.eq('raffles place mrt station')][['latitude', 'longitude']].median()
reference_column = 'dist_raffles_place_mrt_km'
locations[reference_column] = np.nan
if reference.notna().all():
    tree = BallTree(np.radians(reference.to_numpy().reshape(1, 2)), metric='haversine')
    locations.loc[valid_locations, reference_column] = tree.query(location_radians, k=1)[0][:, 0] * EARTH_RADIUS_KM
amenity_features.append(reference_column)

def merge_amenities(frame):
    out = frame.merge(locations[['building_id', *amenity_features]], on='building_id', how='left',
                      sort=False, validate='many_to_one')
    assert np.array_equal(out.row_id, frame.row_id)
    return out

train_enriched = merge_amenities(train_enriched)
test_enriched = merge_amenities(test_enriched)
display(pd.DataFrame(poi_summary))
display(train_enriched[amenity_features].describe().round(3))


,source,input_rows,unique_locations
0,mrt_open_snapshot,231,192
1,mrt_planned_snapshot,12,11
2,mall_snapshot,113,113
3,primary_school_snapshot,182,182


,dist_mrt_open_snapshot_km,count_mrt_open_snapshot_500m,count_mrt_open_snapshot_1000m,dist_mrt_planned_snapshot_km,count_mrt_planned_snapshot_500m,count_mrt_planned_snapshot_1000m,dist_mall_snapshot_km,count_mall_snapshot_500m,count_mall_snapshot_1000m,dist_primary_school_snapshot_km,count_primary_school_snapshot_500m,count_primary_school_snapshot_1000m,dist_raffles_place_mrt_km
count,150000.000,150000.000,150000.000,150000.000,150000.000,150000.000,150000.000,150000.000,150000.000,150000.000,150000.000,150000.000,150000.000
mean,0.591,0.493,1.699,3.387,0.049,0.171,0.810,0.325,1.167,0.427,0.972,2.969,11.499
std,0.327,0.584,1.201,2.553,0.216,0.382,0.447,0.570,1.207,0.262,0.822,1.606,4.651
min,0.022,0.000,0.000,0.062,0.000,0.000,0.000,0.000,0.000,0.024,0.000,0.000,0.580
25%,0.368,0.000,1.000,1.382,0.000,0.000,0.462,0.000,0.000,0.244,0.000,2.000,7.994
50%,0.533,0.000,2.000,2.515,0.000,0.000,0.732,0.000,1.000,0.375,1.000,3.000,12.241
75%,0.756,1.000,2.000,5.157,0.000,0.000,1.083,1.000,2.000,0.544,1.000,4.000,14.924
max,2.572,5.000,10.000,10.397,1.000,2.000,3.202,6.000,12.000,3.309,4.000,10.000,20.228


## 5. Monthly COE Features

The two bidding exercises are first aggregated by category and month, after which historical changes are calculated. The **source month** is retained rather than directly joining these features to the rental month.

Notebook 03 maps each row to previously completed months and freezes information at the forecast origin during each temporal validation, preventing the use of future economic data.


In [14]:
coe = load_raw('sg-coe-prices.csv')
coe.columns = coe.columns.str.lower()
coe['category'] = clean_text(coe.category)
coe['source_month'] = parse_coe_date(
    coe['year'], coe['month']
)
for column in ['quota_premium', 'prevailing_quota_premium', 'quota', 'bids_received']:
    coe[column] = parse_amount(coe[column])
coe['bid_pressure'] = coe.bids_received / coe.quota.where(coe.quota.gt(0))
coe_wide = coe.pivot_table(index='source_month', columns='category',
                          values=['quota_premium', 'bid_pressure'], aggfunc='mean')
coe_wide.columns = [f'macro_coe_{metric}_{category}' for metric, category in coe_wide.columns]
full_months = pd.date_range(coe.source_month.min(), coe.source_month.max(), freq='MS')
coe_wide = coe_wide.reindex(full_months).sort_index()
for category in ['a', 'b', 'c', 'e']:
    column = f'macro_coe_quota_premium_{category}'
    if column in coe_wide:
        coe_wide[f'{column}_change_1m'] = coe_wide[column].pct_change(fill_method=None)
display(coe_wide.tail())


,macro_coe_bid_pressure_a,macro_coe_bid_pressure_b,macro_coe_bid_pressure_c,macro_coe_bid_pressure_e,macro_coe_quota_premium_a,macro_coe_quota_premium_b,macro_coe_quota_premium_c,macro_coe_quota_premium_e,macro_coe_quota_premium_a_change_1m,macro_coe_quota_premium_b_change_1m,macro_coe_quota_premium_c_change_1m,macro_coe_quota_premium_e_change_1m
2026-04-01,1.955336,1.602959,1.765283,1.994284,120505.0,121000.5,81751.0,123001.5,0.094953,0.054149,0.061701,0.055766
2026-05-01,1.717234,1.599471,1.673384,1.794476,124509.5,127868.5,89851.0,128850.0,0.033231,0.05676,0.099081,0.047548
2026-06-01,1.539701,1.411793,1.443141,1.881497,124928.0,125245.5,93500.5,129001.0,0.003361,-0.020513,0.040617,0.001172
2026-07-01,1.393779,1.648634,1.460907,1.671398,127500.0,130389.5,94444.5,129886.0,0.020588,0.041071,0.010096,0.00686
2026-08-01,1.241436,1.421166,1.47619,<NA>,123890.0,129910.0,91545.0,<NA>,-0.028314,-0.003677,-0.030701,<NA>


## 6. Monthly Stock Features and Source Dates

Six predefined `.SI` stocks are used as proxies: three banks, an airline, a telecommunications company, and a real estate investment trust. Their selection is not based on correlations with rent or test results.

Month-end CLOSE values, one-month returns, and three-month returns are calculated separately for each stock. Prices from different stocks are not directly combined into a single index.

Returns are calculated strictly from past months, and missing values are not filled using later observations. Stock prices are not interpreted as causal evidence for changes in rent.


In [17]:
stocks = load_raw('sg-stock-prices.csv')
stocks.columns = stocks.columns.str.lower()
stocks['date'] = pd.to_datetime(stocks.date)
stocks['source_month'] = stocks.date.dt.to_period('M').dt.to_timestamp()
selected_symbols = ['D05.SI', 'O39.SI', 'U11.SI', 'C6L.SI', 'Z74.SI', 'C38U.SI']
selected = stocks[stocks.symbol.isin(selected_symbols)].sort_values('date')
stock_close = selected.groupby(['source_month', 'symbol']).close.last().unstack()
stock_close = stock_close.reindex(pd.date_range(stocks.source_month.min(), stocks.source_month.max(), freq='MS'))
stock_features = pd.DataFrame(index=stock_close.index)
for symbol in selected_symbols:
    if symbol not in stock_close:
        continue
    name = symbol.lower().replace('.', '_')
    stock_features[f'macro_stock_{name}_close'] = stock_close[symbol]
    stock_features[f'macro_stock_{name}_return_1m'] = stock_close[symbol].pct_change(1, fill_method=None)
    stock_features[f'macro_stock_{name}_return_3m'] = stock_close[symbol].pct_change(3, fill_method=None)
macro_monthly = coe_wide.join(stock_features, how='outer').sort_index()
macro_monthly.index.name = 'source_month'
macro_monthly = macro_monthly.replace([np.inf, -np.inf], np.nan)
assert macro_monthly.index.is_unique
display(macro_monthly.tail())
print('Monthly macro features:', macro_monthly.shape)


,macro_coe_bid_pressure_a,macro_coe_bid_pressure_b,macro_coe_bid_pressure_c,macro_coe_bid_pressure_e,macro_coe_quota_premium_a,macro_coe_quota_premium_b,macro_coe_quota_premium_c,macro_coe_quota_premium_e,macro_coe_quota_premium_a_change_1m,macro_coe_quota_premium_b_change_1m,macro_coe_quota_premium_c_change_1m,macro_coe_quota_premium_e_change_1m,macro_stock_d05_si_close,macro_stock_d05_si_return_1m,macro_stock_d05_si_return_3m,macro_stock_o39_si_close,macro_stock_o39_si_return_1m,macro_stock_o39_si_return_3m,macro_stock_u11_si_close,macro_stock_u11_si_return_1m,macro_stock_u11_si_return_3m,macro_stock_c6l_si_close,macro_stock_c6l_si_return_1m,macro_stock_c6l_si_return_3m,macro_stock_z74_si_close,macro_stock_z74_si_return_1m,macro_stock_z74_si_return_3m,macro_stock_c38u_si_close,macro_stock_c38u_si_return_1m,macro_stock_c38u_si_return_3m
source_month,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-04-01,1.955336,1.602959,1.765283,1.994284,120505.0,121000.5,81751.0,123001.5,0.094953,0.054149,0.061701,0.055766,58.50,0.028120,-0.011824,21.90,-0.003186,0.031559,36.15,-0.013643,-0.055396,6.29,-0.046970,-0.009449,4.59,-0.070850,0.000000,2.36,0.030568,-0.012552
2026-05-01,1.717234,1.599471,1.673384,1.794476,124509.5,127868.5,89851.0,128850.0,0.033231,0.05676,0.099081,0.047548,62.84,0.074188,0.100140,23.40,0.068493,0.091927,37.60,0.040111,0.017041,6.78,0.077901,-0.055710,4.34,-0.054466,-0.138889,2.27,-0.038136,-0.073469
2026-06-01,1.539701,1.411793,1.443141,1.881497,124928.0,125245.5,93500.5,129001.0,0.003361,-0.020513,0.040617,0.001172,65.40,0.040738,0.149385,24.79,0.059402,0.128357,39.76,0.057447,0.084857,7.68,0.132743,0.163636,4.41,0.016129,-0.107287,2.37,0.044053,0.034934
2026-07-01,1.393779,1.648634,1.460907,1.671398,127500.0,130389.5,94444.5,129886.0,0.020588,0.041071,0.010096,0.00686,74.85,0.144495,0.279487,29.10,0.173860,0.328767,43.88,0.103622,0.213831,7.61,-0.009115,0.209857,4.58,0.038549,-0.002179,2.49,0.050633,0.055085
2026-08-01,1.241436,1.421166,1.47619,<NA>,123890.0,129910.0,91545.0,<NA>,-0.028314,-0.003677,-0.030701,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Monthly macro features: (68, 30)


## 7. Four Feature Tiers and Field Descriptions

The three groups of static/row-level features are added progressively. For the fourth tier, the training notebook adds macro features within each fold, aligned with the information available at the forecast origin.

| Tier | Contents |
|---|---|
| core | Property attributes + Town + rental date |
| building | core + building information, precise location, and region |
| amenities | building + distances to and counts of transport amenities, everyday amenities, and primary schools |
| macro | amenities + economic proxies available at the forecast origin |

All inputs exclude `monthly_rent`, `row_id`, raw date metadata, and the constant `furnished/fee` fields.
Missing-value handling and encoding are fitted on the relevant training fold immediately before model training.


In [18]:
base_feature_columns = list(dict.fromkeys(core_features + building_features + amenity_features))
categorical_features = [c for c in base_feature_columns if c in [
    'town', 'flat_type', 'flat_model', 'street', 'building_id', 'postal_sector',
    'subzone', 'planning_area', 'region']]
feature_sets = {
    'core': core_features,
    'building': core_features + building_features,
    'amenities': base_feature_columns,
    'macro': base_feature_columns + list(macro_monthly.columns)
}
metadata_columns = ['row_id', 'rent_approval_date']
train_features = train_enriched[metadata_columns + base_feature_columns].copy()
test_features = test_enriched[metadata_columns + base_feature_columns].copy()
for frame in [train_features, test_features]:
    for column in categorical_features:
        frame[column] = frame[column].astype('string')
    numeric = [c for c in base_feature_columns if c not in categorical_features]
    frame[numeric] = frame[numeric].replace([np.inf, -np.inf], np.nan)

assert list(train_features.columns) == list(test_features.columns)
assert 'monthly_rent' not in train_features
assert train_features.row_id.is_unique and test_features.row_id.is_unique
assert np.array_equal(train_features.row_id, targets.row_id)
assert len(train_features) == len(train_raw) and len(test_features) == len(test_raw)
assert set(['furnished', 'fee']).isdisjoint(base_feature_columns)

definitions = {
    'town': 'Canonical town category', 'flat_type': 'Normalized flat type',
    'flat_model': 'Canonical flat model', 'floor_area_sqm': 'Floor area in square metres',
    'lease_commence_date': 'Lease commencement year', 'approval_year': 'Approval year',
    'approval_month': 'Approval calendar month', 'month_index': 'Months since January 2021',
    'month_sin': 'Sine of calendar month', 'month_cos': 'Cosine of calendar month',
    'lease_age': 'Approval year minus lease commencement year',
    'lease_age_invalid': 'Indicator of negative lease age', 'log_floor_area': 'log(1 + floor area)',
    'flat_room_number': 'n in n-room; missing for executive', 'street': 'Canonical street',
    'building_id': 'Town|block|street joint identifier', 'postal_sector': 'First two postal-code digits',
    'postal_sector_imputed': 'Indicator of postal sector imputed from the unique known sector within the same town and street',
    'latitude': 'Block latitude', 'longitude': 'Block longitude', 'max_floor': 'Maximum floor of block, not unit floor',
    'year_completed': 'Block completion year', 'building_age': 'Approval year minus completion year',
    'building_age_invalid': 'Indicator of negative building age', 'subzone': 'Subzone category',
    'planning_area': 'Planning-area category', 'region': 'Region category',
    'block_metadata_missing': 'Indicator of an unmatched block lookup',
    'dist_raffles_place_mrt_km': 'Haversine distance to the named MRT reference in the provided file'
}
dictionary = []
for group, columns in [('core', core_features), ('building', building_features),
                       ('amenities', amenity_features), ('macro', list(macro_monthly.columns))]:
    for column in columns:
        description = definitions.get(column)
        if description is None:
            description = ('Distance in km to nearest named snapshot location' if column.startswith('dist_')
                else 'Number of unique named snapshot locations within the stated radius' if column.startswith('count_')
                else 'Monthly external proxy; source month aligned inside each training fold')
        dictionary.append({'feature': column, 'group': group,
            'type': 'categorical' if column in categorical_features else 'numeric', 'description': description})
feature_dictionary = pd.DataFrame(dictionary)
display(pd.DataFrame({'feature_set': list(feature_sets), 'feature_count': [len(v) for v in feature_sets.values()]}))
display(feature_dictionary)


,feature_set,feature_count
0,core,14
1,building,28
2,amenities,41
3,macro,71


,feature,group,type,description
0,town,core,categorical,Canonical town category
1,flat_type,core,categorical,Normalized flat type
2,flat_model,core,categorical,Canonical flat model
3,floor_area_sqm,core,numeric,Floor area in square metres
4,lease_commence_date,core,numeric,Lease commencement year
...,...,...,...,...
66,macro_stock_z74_si_return_1m,macro,numeric,Monthly external proxy; source month aligned i...
67,macro_stock_z74_si_return_3m,macro,numeric,Monthly external proxy; source month aligned i...
68,macro_stock_c38u_si_close,macro,numeric,Monthly external proxy; source month aligned i...
69,macro_stock_c38u_si_return_1m,macro,numeric,Monthly external proxy; source month aligned i...


## 8. Saving Features, Labels, and Reproducibility Information

Input-file hashes identify the version of the raw data. Generated outputs do not modify the original CSV files. Rerunning this notebook overwrites derived files with the same names within this project.


In [19]:
train_features.to_csv(PROCESSED / 'train_features.csv', index=False)
test_features.to_csv(PROCESSED / 'test_features.csv', index=False)
targets.to_csv(PROCESSED / 'train_targets.csv', index=False)
macro_monthly.to_csv(PROCESSED / 'macro_monthly.csv')
feature_dictionary.to_csv(PROCESSED / 'feature_dictionary.csv', index=False)
metadata = {
    'feature_pipeline_version': '1.0.0', 'train_rows': len(train_features), 'test_rows': len(test_features),
    'feature_sets': feature_sets, 'categorical_features': categorical_features,
    'macro_columns': list(macro_monthly.columns), 'metadata_columns': metadata_columns,
    'constant_raw_columns': constant_raw_columns, 'duplicate_policy': 'retain',
    'poi_policy': 'provided snapshot; no historical opening dates available',
    'default_macro_policy': 'fixed_origin', 'selected_stock_symbols': selected_symbols,
    'train_last_month': train_features.rent_approval_date.max().strftime('%Y-%m'),
    'test_last_month': test_features.rent_approval_date.max().strftime('%Y-%m'),
    'source_sha256': {name: file_sha256(raw_path(name)) for name in FILES}
}
(PROCESSED / 'feature_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved feature files to:', PROCESSED)
print('Training feature shape:', train_features.shape, '| Test feature shape:', test_features.shape)


Saved feature files to: d:\AppData\NUS\Sem1\5228 Knowledge Discovery and Data Mining\cs5228-hdb-rent-prediction\data\processed
Training feature shape: (150000, 43) | Test feature shape: (50000, 43)
